In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pyngrok xgboost scikit-learn joblib flask -q
!ngrok authtoken 38mjJe1niAD5rUmOfxqkHyAojmR_qTCJXiRnisFM36Vp3pVS

from flask import Flask, render_template, request
from pyngrok import ngrok
import joblib
import numpy as np
import threading
import csv
import os
from datetime import datetime

# Load Brinda's trained model
model    = joblib.load('/content/drive/MyDrive/Colab Notebooks/final_fraud_model.pkl')
scaler   = joblib.load('/content/drive/MyDrive/Colab Notebooks/scaler.pkl')
features = joblib.load('/content/drive/MyDrive/Colab Notebooks/feature_list.pkl')

DYNAMIC_DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/dynamic_transactions.csv'
template_folder   = '/content/drive/MyDrive/Colab Notebooks/template'
app = Flask(__name__, template_folder=template_folder)

@app.route("/")
@app.route("/home")
def home():
    return render_template("home.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        # Read simple inputs from form
        amount        = float(request.form.get('Transaction_Amount', 1000))
        hour_cat      = request.form.get('hour_category', 'morning')
        device_cat    = request.form.get('device_category', 'mobile')
        txn_type_cat  = request.form.get('txn_type_category', 'payment')
        location      = int(request.form.get('location_category', 2))
        payment_cat   = request.form.get('payment_category', 'upi')
        device_change = int(request.form.get('device_change', 0))
        txn_freq      = request.form.get('txn_frequency', 'low')
        prev_fraud    = int(request.form.get('prev_fraud', 0))

        # Convert to model features
        hour_map     = {'morning':9, 'afternoon':14, 'evening':19, 'night':2}
        device_map   = {'mobile':0, 'desktop':1, 'tablet':2}
        txn_type_map = {'transfer':0, 'payment':1, 'withdrawal':2}
        payment_map  = {'upi':0, 'netbanking':1, 'card':2}
        freq_map     = {'low':2, 'medium':6, 'high':12, 'very_high':20}

        hour         = hour_map.get(hour_cat, 9)
        night_txn    = 1 if hour >= 22 or hour < 6 else 0
        device_used  = device_map.get(device_cat, 0)
        txn_type     = txn_type_map.get(txn_type_cat, 1)
        payment      = payment_map.get(payment_cat, 0)
        num_txn_24h  = freq_map.get(txn_freq, 2)

        user_id       = np.random.randint(1000, 9999)
        account_age   = np.random.randint(180, 1800)
        txn_count     = np.random.randint(5, 50)
        user_avg_amt  = round(np.random.uniform(500, 5000), 2)
        amt_deviation = round(amount - user_avg_amt, 2)

        feature_values = {
            'User_ID':                          user_id,
            'Transaction_Amount':               amount,
            'Transaction_Type':                 txn_type,
            'Device_Used':                      device_used,
            'Location':                         location,
            'Previous_Fraudulent_Transactions': prev_fraud,
            'Account_Age':                      account_age,
            'Number_of_Transactions_Last_24H':  num_txn_24h,
            'Payment_Method':                   payment,
            'Hour':                             hour,
            'txn_count_user':                   txn_count,
            'user_avg_amt':                     user_avg_amt,
            'amt_deviation':                    amt_deviation,
            'night_txn':                        night_txn,
            'device_change':                    device_change,
        }

        # Run through model
        input_array  = np.array([feature_values[f] for f in features]).reshape(1,-1)
        input_scaled = scaler.transform(input_array)

        prediction  = model.predict(input_scaled)[0]
        probability = model.predict_proba(input_scaled)[0][1]
        result      = "⚠️ FRAUD DETECTED" if prediction == 1 else "✅ LEGITIMATE TRANSACTION"

        # Save to dynamic dataset
        row = dict(feature_values)
        row['Fraudulent']         = int(prediction)
        row['Fraud_Probability']  = round(float(probability), 4)
        row['Timestamp']          = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        file_exists = os.path.isfile(DYNAMIC_DATA_PATH)
        with open(DYNAMIC_DATA_PATH, 'a', newline='') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=row.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)

        return render_template("home.html",
                               result=result,
                               probability=round(float(probability)*100, 2))
    except Exception as e:
        return render_template("home.html", error=str(e))

# Start app
public_url = ngrok.connect(5000)
print("=" * 50)
print(f"🌐 Your app is live at: {public_url}")
print("=" * 50)
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🌐 Your app is live at: NgrokTunnel: "https://uninquisitorial-ovarian-zoe.ngrok-free.dev" -> "http://localhost:5000"
